# German Institutions Analysis

In [1]:
import pandas as pd
from google.cloud import bigquery

## Download data

- Get all institutions from germany and their published scientific documents

In [2]:
def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

In [6]:
sql = """select ins.DISPLAY_NAME, ins.TYPE, COUNT(DISTINCT(w.ID)) AS work_count
         FROM """ + PROJECT_ID + "." + DATASET_ID + ".works" + " w" """
         JOIN """ + PROJECT_ID + "." + DATASET_ID + ".works_authorships" + """ wauth ON wauth.WORK_ID = w.ID
         JOIN """ + PROJECT_ID + "." + DATASET_ID + ".institutions" + """ ins ON ins.ID = wauth.INSTITUTION_ID
         WHERE ins.COUNTRY_CODE = 'DE' AND w.PUBLICATION_YEAR > 1999 AND w.PUBLICATION_YEAR < 2021
         GROUP BY ins.DISPLAY_NAME, ins.TYPE ORDER BY work_count DESC"""
df = bg_query(sql)
df.to_csv('../data/interim/institutions_work_count.csv', index = False)
df

/home/siris/DI_proj_germany/venv/lib/python3.10/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/DI_proj_germany/venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DISPLAY_NAME,TYPE,work_count
0,Max Planck Society,nonprofit,169201
1,Heidelberg University,funder,157820
2,Ludwig-Maximilians-Universität München,funder,124978
3,Technical University of Munich,funder,114768
4,Universität Hamburg,funder,105689
...,...,...,...
4599,Species Conservation Foundation,other,1
4600,Institut für Internationale Kommunikation,nonprofit,1
4601,Stiftung Tumorforschung Kopf-Hals,nonprofit,1
4602,HDG Bavaria (Germany),company,1


**Manual reconciliation of names using open - refine + manual validation**

- From 4602 initial organizations we get a number of xxx which merge spelling errors
- From those xxx organizations we merge the parent ones in order to only have the main achieving a set of xxx
- We manually classify them according to the type of institution respect the achieved funding 